In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, HTML

BASE_DIR = Path(".").resolve()
MAPPINGS_DIR = BASE_DIR / "acc_mappings" / "acc_mappings_isx"


In [2]:

rows = []

for fp in sorted(MAPPINGS_DIR.glob("*.json")):
    firm_id = fp.stem
    data = json.loads(fp.read_text(encoding="utf-8"))

    for v in data.get("variables", []):
        if v.get("needs_manual_review", False):
            final_choice = v.get("final_choice", [])
            # make final_choice readable
            final_str = "; ".join([f"{x['sheet_name']}::{x['row_label']}" for x in final_choice]) if final_choice else ""
            rows.append({
                "firm_id": firm_id,
                "variable": v.get("variable", ""),
                "notes": v.get("notes", ""),
                "final_choice": final_str,
                "num_candidates": len(v.get("candidates", [])),
            })

if len(rows) == 0:
    print("No manual reviews needed. All mappings have been reviewed and finalized.")
else:
    df_review = pd.DataFrame(rows).sort_values(["firm_id", "variable"]).reset_index(drop=True)

    print(f"Manual review needed for {len(df_review)} firm-variable mappings across {df_review['firm_id'].nunique() if len(df_review) else 0} firms.")
    display(HTML(f"""
    <div style="max-height: 450px; overflow-y: auto; border: 1px solid #ddd; padding: 6px;">
    {df_review.to_html(index=False)}
    </div>
    """))

Manual review needed for 29 firm-variable mappings across 19 firms.


firm_id,variable,notes,final_choice,num_candidates
ALVO.IC,OANCF,Candidate list for OANCF was empty even though the correct cash flow label exists in the A-column preview. Selected the exact row from the full sheet preview per instructions.,Cash Flow::Net cash from (used in) operating activities,1
AMRQM.IC,PPEGT,"Candidate list appears to miss the best total/net PPE label. 'Capital assets' is the clearest balance sheet total PPE-style line and is preferred over partial gross component rows, but because it was not in the candidate list, manual review is required. | Escape-hatch used (final_choice outside candidate list): [Balance Sheet] Capital assets",Balance Sheet::Capital assets,4
AMRQM.IC,STD,"No STD candidates were provided. Chose current financing debt lines from the balance sheet that appear within current liabilities and exclude leases. 'Loans payable' is the clearest short-term debt line. 'Convertible notes' may also represent current debt in some periods, but classification/currentity across years is less certain from the preview alone, so manual review is needed to confirm whether both should be included or treated as period-specific alternatives.",Balance Sheet::Loans payable; Balance Sheet::Convertible notes,0
BERAHF.IC,STD,"Candidate list appears incomplete. A cleaner minimal set exists in the A-column preview: 'Interest-bearing debts - short-term' plus 'Current portion of long-term interest-bearing debts excluding leases'. This best captures short-term debt while excluding leases and avoids using 'Interest Payable - Short-Term Debt', which is not debt principal. Manual review flagged because final labels were taken from the full preview rather than the provided candidates. | Escape-hatch used (final_choice outside candidate list): [Balance Sheet] Interest-bearing debts - short-term, [Balance Sheet] Current portion of long-term interest-bearing debts excluding leases",Balance Sheet::Interest-bearing debts - short-term; Balance Sheet::Current portion of long-term interest-bearing debts excluding leases,2
BRIMH.IC,CHE,"Candidate list does not include the balance-sheet cash line. 'Bank Deposits' matches the cash balance and ties to cash flow ending cash values, but it is outside the provided candidates. | Escape-hatch used (final_choice outside candidate list): [Balance Sheet] Bank Deposits",Balance Sheet::Bank Deposits,7
BRIMH.IC,OANCF,Candidate list misses the explicit net operating cash flow subtotal. 'As Reported Cash fr Operating Activities' is the best exact match for OANCF. | Escape-hatch used (final_choice outside candidate list): [Cash Flow] As Reported Cash fr Operating Activities,Cash Flow::As Reported Cash fr Operating Activities,5
BRIMH.IC,PPEGT,"Preferred a balance-sheet PPE stock measure. Candidate list misses the cleaner net PPE-style line 'Net, Tangibles', which appears to be the reported net tangible fixed assets/PPE aggregate. Included 'Total Property/Plant/Equipment, Gross' as lower-priority fallback because it is the explicit PPE label but blank in the preview. | Escape-hatch used (final_choice outside candidate list): [Balance Sheet] Net, Tangibles","Balance Sheet::Net, Tangibles; Balance Sheet::Total Property/Plant/Equipment, Gross",5
BRIMH.IC,STD,Candidate list appears incomplete. Best IFRS-style mapping for non-lease current debt is the sum of 'Current Portion of Long-Term Debt excluding Capitalized Leases' and 'Short-Term Debt'. Avoided 'Current LT Debt' because the explicit excluding-leases line suggests the broader line may include leases; avoided 'Liabilities due within 1 Year' because it is a maturity disclosure and may overlap/incompletely capture current borrowings. | Escape-hatch used (final_choice outside candidate list): [Balance Sheet] Current Portion of Long-Term Debt excluding Capitalized Leases,Balance Sheet::Current Portion of Long-Term Debt excluding Capitalized Leases; Balance Sheet::Short-Term Debt,3
EIKF.IC,STD,Candidate list appears to miss the best available balance-sheet